In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1. load data
df = pd.read_csv("../../data/austria/raw/austria_modeling_master.csv", index_col=0, parse_dates=True)

# 2. Split
split_idx = int(len(df) * 0.8)
train_df, test_df = df.iloc[:split_idx], df.iloc[split_idx:]

# 3. features and target definition
target = "target_load_24h"
features = [c for c in df.columns if c != target]

# 4. skaling
scaler_x = StandardScaler()
scaler_y = StandardScaler()

train_df_sc = train_df.copy()
test_df_sc = test_df.copy()

train_df_sc[features] = scaler_x.fit_transform(train_df[features])
test_df_sc[features] = scaler_x.transform(test_df[features])

train_df_sc[[target]] = scaler_y.fit_transform(train_df[[target]])
test_df_sc[[target]] = scaler_y.transform(test_df[[target]])

# 5. Windowing
def create_sequences(df_scaled, feature_cols, target_col, window=168):
    X, y = [], []
    data = df_scaled[feature_cols].values
    labels = df_scaled[target_col].values
    for i in range(len(df_scaled) - window):
        X.append(data[i:i+window])
        y.append(labels[i+window])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_df_sc, features, target)
X_test, y_test = create_sequences(test_df_sc, features, target)